# Model 04: population growth and carrying capacity

This notebook tests the claim that a founder lineage and the rest of a population should be treated as two competing exponential populations. They should not: after intermarriage, the two descendant sets overlap.

The model also allows exponential growth, logistic carrying capacity, bottlenecks, and unequal reproductive weighting.

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Evolution-Creation'):
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git','/content/Evolution-Creation'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e','/content/Evolution-Creation'], check=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from evolution_creation.demography import (
    apply_population_bottleneck,
    constant_population_schedule,
    deterministic_ancestry_fraction_curve,
    exponential_population_schedule,
    logistic_population_schedule,
    simulate_demographic_replicates,
    simulate_overlapping_ancestry_sets,
)


## Controls

The founder ancestry weight is a sensitivity parameter. A value of 1 means neutral reproduction. Values above or below 1 impose a reproductive advantage or disadvantage on founder-descended parents.

In [ ]:
initial_population=widgets.IntSlider(value=1000,min=50,max=5000,step=50,description='Initial N')
generations=widgets.IntSlider(value=20,min=5,max=60,step=1,description='Generations')
growth_model=widgets.Dropdown(options=['constant','exponential','logistic'],value='logistic',description='Growth')
growth_rate=widgets.FloatSlider(value=0.20,min=0.0,max=0.5,step=0.01,description='Growth r')
carrying_capacity=widgets.IntSlider(value=5000,min=100,max=20000,step=100,description='Capacity K')
founders=widgets.IntSlider(value=1,min=1,max=50,step=1,description='Founders')
ancestry_weight=widgets.FloatSlider(value=1.0,min=0.2,max=3.0,step=0.1,description='Ancestry w')
use_bottleneck=widgets.Checkbox(value=False,description='Bottleneck')
bottleneck_start=widgets.IntSlider(value=6,min=1,max=50,description='Start')
bottleneck_duration=widgets.IntSlider(value=3,min=1,max=15,description='Duration')
bottleneck_size=widgets.IntSlider(value=100,min=2,max=2000,step=10,description='Bottleneck N')
replicates=widgets.IntSlider(value=200,min=20,max=1000,step=20,description='Replicates')
seed=widgets.IntText(value=20260920,description='Seed')
display(initial_population,generations,growth_model,growth_rate,carrying_capacity,founders,ancestry_weight,use_bottleneck,bottleneck_start,bottleneck_duration,bottleneck_size,replicates,seed)

In [ ]:
def make_schedule():
    n0=initial_population.value
    g=generations.value
    if growth_model.value=='constant':
        schedule=constant_population_schedule(n0,g)
    elif growth_model.value=='exponential':
        schedule=exponential_population_schedule(n0,growth_rate.value,g)
    else:
        k=max(carrying_capacity.value,n0)
        schedule=logistic_population_schedule(n0,k,growth_rate.value,g)
    if use_bottleneck.value:
        schedule=apply_population_bottleneck(
            schedule,
            start_generation=min(bottleneck_start.value,g),
            duration=bottleneck_duration.value,
            bottleneck_size=bottleneck_size.value,
        )
    return schedule

def run_model(_=None):
    schedule=make_schedule()
    founder_count=min(founders.value,schedule[0])
    curves=simulate_demographic_replicates(
        schedule,
        founder_count=founder_count,
        ancestry_parent_weight=ancestry_weight.value,
        replicates=replicates.value,
        seed=seed.value,
    )
    x=np.arange(len(schedule))
    deterministic=deterministic_ancestry_fraction_curve(
        founder_count/schedule[0],len(schedule)-1,ancestry_weight.value
    )

    fig,ax=plt.subplots(figsize=(9,4))
    ax.plot(x,schedule)
    ax.set(xlabel='Generation',ylabel='Population size',title='Demographic schedule')
    plt.show()

    median=np.median(curves,axis=0)
    lo=np.quantile(curves,.10,axis=0)
    hi=np.quantile(curves,.90,axis=0)
    fig,ax=plt.subplots(figsize=(9,5))
    ax.fill_between(x,lo,hi,alpha=.2,label='10th-90th percentile')
    ax.plot(x,median,label='simulation median')
    ax.plot(x,deterministic,'--',label='deterministic expectation')
    ax.set(xlabel='Generation',ylabel='Founder-descendant fraction',ylim=(0,1.02))
    ax.legend(); plt.show()

    final=curves[:,-1]
    print(f'Lineage extinct by final generation: {(final==0).mean():.1%}')
    print(f'Genealogical fixation by final generation: {(final==1).mean():.1%}')

    overlap=simulate_overlapping_ancestry_sets(schedule,founder_count=max(1,min(founder_count,schedule[0]-1)),seed=seed.value)
    fig,ax=plt.subplots(figsize=(9,5))
    ax.plot(x,overlap.founder_descendant_fraction,label='descendants of founder group')
    ax.plot(x,overlap.background_descendant_fraction,label='descendants of background group')
    ax.plot(x,overlap.both_fraction,label='descendants of both')
    ax.set(xlabel='Generation',ylabel='Fraction of population',ylim=(0,1.02))
    ax.legend(); plt.show()

button=widgets.Button(description='Run simulation',button_style='primary')
button.on_click(run_model)
display(button)
run_model()

## Interpretation

The founder-descendant set and background-descendant set overlap after intermarriage. Their fractions should not be added as if they were disjoint populations. Carrying capacity controls total population size; under neutral random mating it does not by itself create a relative advantage for one ancestry class.